# OSC AlphaGenome setup
## imports

In [16]:
from alphagenome_research.model import dna_model
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript
from alphagenome.data import ontology
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
from pysam import VariantFile
import pysam
from io import StringIO
from tqdm import tqdm
import os
import gc
from cyvcf2 import VCF

# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='true'
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.9' # pre allocates XX% of total GPU instead of the default 75%

pd.set_option('display.max_columns', None)

## common variables

In [5]:
LMNA_START = 156_082_572
LMNA_END = 156_140_081
gene_symbol = "LMNA"
LMNA_INTERVAL = genome.Interval('chr1', 156_082_572, 156_140_081)


BASE_PATH = '/users/PAS2905/coraalbers/'
AG_DATA_PATH = '/users/PAS2905/coraalbers/ag/ag_data/'

HG38_FASTA_PATH = '/users/PAS2905/coraalbers/ag/hg38.fa'
HG38_GTF_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.annotation.gtf.gz.feather'
HG38_SPLICE_START_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_starts.feather'
HG38_SPLICE_END_PATH = '/users/PAS2905/coraalbers/ag/ag_data/gencode.v46.splice_sites_ends.feather'

CLINVAR_PATH = '/users/PAS2905/coraalbers/ag/clinvar.vcf.gz'

gtf = pd.read_feather( 'https://storage.googleapis.com/alphagenome/reference/gencode/' 'hg38/gencode.v46.annotation.gtf.gz.feather' )

output_modalities = ['atac',	
    'cage',	
    'chip_histone',	
    'chip_tf',	
    'contact_maps',	
    'dnase',	
    'procap',	
    'rna_seq',	
    'splice_junctions',	
    'splice_site_usage',	
    'splice_sites']

requested_outputs = {dna_client.OutputType.ATAC,
        dna_client.OutputType.CAGE,
        dna_client.OutputType.DNASE,
        dna_client.OutputType.PROCAP,
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.SPLICE_SITES,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
        dna_client.OutputType.CONTACT_MAPS,
        dna_client.OutputType.CHIP_HISTONE,
        dna_client.OutputType.CHIP_TF}

## model initialization

In [3]:
model = dna_model.create_from_huggingface( 
    'all_folds', 
    organism_settings={ 
        dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings( 
            fasta_path=HG38_FASTA_PATH, 
            gtf_feather_path=HG38_GTF_PATH, 
            splice_site_starts_feather_path=HG38_SPLICE_START_PATH, 
            splice_site_ends_feather_path=HG38_SPLICE_END_PATH, 
        ), dna_model.Organism.MUS_MUSCULUS: dna_model.OrganismSettings() } )

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

# import ClinVar database

In [6]:
gene_symbol = "LMNA"
window_bp = 1_000_000

gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
region = gene_interval.pad(window_bp, window_bp)
vcf_contig = region.chromosome.removeprefix("chr")

vcf_path = CLINVAR_PATH
with VariantFile(vcf_path) as vcf_in:
    nearby = [
        rec for rec in vcf_in.fetch(
            vcf_contig,
            max(region.start - 1, 0),
            region.end,
        )
    ]
print(f"{len(nearby)} variants within ±{window_bp:,} bp of {gene_symbol}")
print(f"Region: {region.chromosome}:{region.start}-{region.end}")

13076 variants within ±1,000,000 bp of LMNA
Region: chr1:155082571-157140081


In [7]:
def variants_near_gene(vcf_path, gene_symbol, gtf, window_bp=500_000):
    region = gene_interval.pad(window_bp, window_bp)
    contig = to_vcf_contig(region.chromosome)
    gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
    with VariantFile(vcf_path) as vcf_in:
        # nearby = [
        #     rec for rec in vcf_in.fetch(
        #         vcf_contig,
        #         max(region.start - 1, 0),
        #         region.end,
        #     )
        # ]
        # print(f"{len(nearby)} variants within ±{window_bp:,} bp of {gene_symbol}")
        # print(f"Region: {region.chromosome}:{region.start}-{region.end}")

        for record in vcf_in.fetch(contig, max(region.start - 1, 0), region.end):
            yield record
        

In [8]:
variants_near_gene(vcf_path, 'LMNA', gtf, 500_000)

<generator object variants_near_gene at 0x153d4fb0d340>

In [15]:
# Open the input VCF/BCF file
input_vcf = CLINVAR_PATH
output_vcf = "outputs/clinvar_LMNA.PLP.vcf"

PATHOGENIC = {
    "Pathogenic",
    "Likely_pathogenic",
    "Pathogenic/Likely_pathogenic"    
}

with pysam.VariantFile(input_vcf) as infile:
    # Copy the header structure for the output file
    header = infile.header
    
    # Open the output file in write mode ('w' for VCF, 'wb' for BCF)
    with pysam.VariantFile(output_vcf, "w", header=header) as outfile:
        
        # Iterate over each variant record sequentially
        for record in infile:
            
            # clnsig = record.info.get("CLNSIG")
            # if clnsig is not None:
            #     continue
            
            # pysam returns a tuple for Number=. fields
            if "LMNA:" in record.info.get("GENEINFO", ""):
                # continue
                if isinstance(clnsig, tuple):
                    # continue
                    for sig in PATHOGENIC:
                        if sig in clnsig:
                            continue
                    # return any(sig in PATHOGENIC for sig in clnsig)
                        
            # Write the record to the new file if it passes all criteria
            outfile.write(record)


In [29]:
vcf = VCF(CLINVAR_PATH)

PATHOGENIC = {
    "Pathogenic",
    "Likely_pathogenic",
    "Pathogenic/Likely_pathogenic"    
}

for v in vcf('1:156082572-156140081'):
    # print(v)
    clnsig = v.INFO.get("CLNSIG")
    # print(clnsig)
    if 'LMNA:' in v.INFO["GENEINFO"] and clnsig is not None:
        if any(item in v.INFO["CLNSIG"] for item in PATHOGENIC):
            print(str(v))


1	156114912	66757	GCCGGCCATGGAGACC	G	.	.	ALLELEID=77654;CLNDISDB=MedGen:CN380145|MedGen:C3661900;CLNDN=LMNA-related_disorder|not_provided;CLNHGVS=NC_000001.11:g.156114916_156114930del;CLNREVSTAT=criteria_provided,_single_submitter;CLNSIG=Pathogenic;CLNSIGSCV=SCV006336779;CLNVC=Deletion;CLNVCSO=SO:0000159;CLNVI=ClinGen:CA217675;GENEINFO=LMNA:4000;MC=SO:0001582|initiator_codon_variant,SO:0001822|inframe_deletion;ORIGIN=1;RS=267607546

1	156114913	2690636	CCGGCCATGGAGACCCCGTCCCAG	C	.	.	ALLELEID=2851669;CLNDISDB=MedGen:C3661900;CLNDN=not_provided;CLNHGVS=NC_000001.11:g.156114918_156114940del;CLNREVSTAT=criteria_provided,_single_submitter;CLNSIG=Likely_pathogenic;CLNSIGSCV=SCV004238270;CLNVC=Deletion;CLNVCSO=SO:0000159;CLNVI=ClinGen:CA2697554571;GENEINFO=LMNA:4000|LOC129931597:129931597;MC=SO:0001582|initiator_codon_variant,SO:0001589|frameshift_variant,SO:0001623|5_prime_UTR_variant,SO:0001627|intron_variant;ORIGIN=1;RS=2527828682

1	156114919	3229277	A	G	.	.	ALLELEID=3391194;CLNDISDB=MedG

In [18]:
vcf